# Representational Settling Depth (RSD)
## Pre-Inference Query Difficulty Estimator for LLM Routing

This notebook implements the RSD method step by step.

**Pipeline:**
1. Setup & Imports
2. Load Routing LLM with Hidden State Hooks
3. Extract Layer-wise Hidden States
4. Compute Inter-layer Cosine Similarities
5. Detect Settling Layer with Early Termination
6. Compute Normalized RSD Score
7. Calibrate Thresholds (τ and δ)
8. Route Query Based on RSD Score
9. Evaluate on Benchmark (RouterBench-style)
10. Ablation Studies
11. Visualize Results

---
## Cell 1 — Install Dependencies

In [ ]:
# Run once. Restart kernel after installation.
!pip install transformers torch accelerate scipy scikit-learn matplotlib seaborn datasets tqdm --quiet

---
## Cell 2 — Imports & Global Configuration

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

# ── Global configuration ───────────────────────────────────────────────────────
@dataclass
class RSDConfig:
    """
    All tunable parameters in one place.
    Change values here; the rest of the notebook reads from this object.
    """
    # Model
    routing_model_name: str     = "meta-llama/Llama-3.2-1B-Instruct"  # swap freely
    device: str                  = "cuda" if torch.cuda.is_available() else "cpu"
    torch_dtype: torch.dtype     = torch.float16

    # RSD core parameters
    tau: float                   = 0.97   # settling threshold (grid-searched in Cell 8)
    delta: float                 = 0.5    # routing threshold (calibrated in Cell 9)
    l_min_frac: float            = 0.10   # minimum depth floor as fraction of L

    # Calibration grid
    tau_grid: List[float]        = field(default_factory=lambda:
                                         [0.90, 0.92, 0.94, 0.96, 0.97, 0.98, 0.99])
    delta_grid: List[float]      = field(default_factory=lambda:
                                         list(np.linspace(0.1, 0.9, 17)))

    # Evaluation
    strong_model_quality: float  = 1.0    # assumed oracle quality of strong model
    weak_model_quality: float    = 0.65   # assumed quality of weak model on hard queries


CFG = RSDConfig()
print(f"Device : {CFG.device}")
print(f"Model  : {CFG.routing_model_name}")
print(f"τ      : {CFG.tau}")
print(f"δ      : {CFG.delta}")
print(f"l_min  : {CFG.l_min_frac * 100:.0f}% of L")

---
## Cell 3 — Module: Model Loader
Loads the routing LLM and registers forward hooks to capture hidden states.

In [ ]:
class RoutingModel:
    """
    Wraps a decoder-only HuggingFace LLM and captures the final-token
    hidden state at every transformer layer via forward hooks.

    Usage
    -----
        model = RoutingModel(CFG)
        hidden_states = model.extract_hidden_states("What is 2+2?")
        # hidden_states: List[torch.Tensor] of shape (d,), one per layer
    """

    def __init__(self, cfg: RSDConfig):
        self.cfg = cfg
        self._hidden_states: List[torch.Tensor] = []
        self._hooks = []

        print(f"Loading tokenizer from {cfg.routing_model_name} ...")
        self.tokenizer = AutoTokenizer.from_pretrained(
            cfg.routing_model_name,
            trust_remote_code=True
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        print(f"Loading model ...")
        self.model = AutoModelForCausalLM.from_pretrained(
            cfg.routing_model_name,
            torch_dtype=cfg.torch_dtype,
            device_map="auto",
            trust_remote_code=True,
            output_hidden_states=True   # ensures HuggingFace returns all hidden states
        )
        self.model.eval()

        # Total number of transformer blocks (excluding embedding layer)
        self.num_layers: int = self.model.config.num_hidden_layers
        self.l_min: int = max(1, int(cfg.l_min_frac * self.num_layers))
        print(f"Model loaded | Layers: {self.num_layers} | l_min: {self.l_min}")

    # ── Hidden state extraction ────────────────────────────────────────────────

    @torch.no_grad()
    def extract_hidden_states(self, query: str) -> List[torch.Tensor]:
        """
        Run a full forward pass and return the final-token hidden state
        at every layer l = 0 … L.

        Returns
        -------
        List[torch.Tensor]  length = L+1 (embedding layer + L transformer blocks)
                            each tensor shape: (hidden_dim,)
        """
        inputs = self.tokenizer(
            query,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(self.cfg.device)

        outputs = self.model(**inputs)

        # outputs.hidden_states is a tuple: (embedding, layer_1, ..., layer_L)
        # We take the last token position [-1] and detach to CPU for efficiency.
        hidden_states = [
            hs[0, -1, :].float().cpu()          # shape: (hidden_dim,)
            for hs in outputs.hidden_states      # L+1 tensors
        ]
        return hidden_states

    @torch.no_grad()
    def extract_hidden_states_batch(
        self, queries: List[str]
    ) -> List[List[torch.Tensor]]:
        """
        Batched version of extract_hidden_states.
        Pads queries to the same length within the batch.

        Returns
        -------
        List of hidden_state lists, one per query.
        """
        inputs = self.tokenizer(
            queries,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(self.cfg.device)

        outputs = self.model(**inputs)

        # For each query in the batch, find the last non-padding token position.
        attention_mask = inputs["attention_mask"]          # (B, seq_len)
        last_token_idx = attention_mask.sum(dim=1) - 1    # (B,)

        batch_hidden_states = []
        for b in range(len(queries)):
            idx = last_token_idx[b].item()
            hs_per_layer = [
                hs[b, idx, :].float().cpu()
                for hs in outputs.hidden_states
            ]
            batch_hidden_states.append(hs_per_layer)

        return batch_hidden_states


# Instantiate — comment out if running without a GPU/model for demo mode
# routing_model = RoutingModel(CFG)
print("RoutingModel class defined. Instantiate with:  routing_model = RoutingModel(CFG)")

---
## Cell 4 — Module: Inter-layer Cosine Similarity
Computes `r_l = cos(h_l, h_{l+1})` for each consecutive layer pair.

In [ ]:
class SimilarityComputer:
    """
    Computes inter-layer cosine similarity from a list of hidden states.

    r_l = cos(h_l, h_{l+1})  for l = 0 … L-1
    """

    @staticmethod
    def compute(hidden_states: List[torch.Tensor]) -> np.ndarray:
        """
        Parameters
        ----------
        hidden_states : List[Tensor]  length L+1, each shape (d,)

        Returns
        -------
        np.ndarray of shape (L,)  — one similarity value per consecutive pair
        """
        similarities = []
        for l in range(len(hidden_states) - 1):
            h_l   = hidden_states[l]
            h_lp1 = hidden_states[l + 1]

            # Cosine similarity: dot product divided by product of norms
            norm_l   = h_l.norm()
            norm_lp1 = h_lp1.norm()

            if norm_l < 1e-8 or norm_lp1 < 1e-8:
                # Degenerate hidden state (near-zero norm) — treat as maximally similar
                similarities.append(1.0)
            else:
                cos_sim = torch.dot(h_l, h_lp1) / (norm_l * norm_lp1)
                similarities.append(cos_sim.item())

        return np.array(similarities, dtype=np.float32)

    @staticmethod
    def compute_batch(
        batch_hidden_states: List[List[torch.Tensor]]
    ) -> List[np.ndarray]:
        """
        Vectorized batch computation.

        Returns
        -------
        List of similarity arrays, one per query.
        """
        return [
            SimilarityComputer.compute(hs)
            for hs in batch_hidden_states
        ]


# ── Quick unit test with random tensors ────────────────────────────────────────
def _test_similarity_computer():
    L, d = 32, 2048
    # Simulate: easy query (slowly changing hidden states)
    base = torch.randn(d)
    easy_hs = [base + 0.01 * torch.randn(d) for _ in range(L + 1)]
    # Simulate: hard query (rapidly changing hidden states)
    hard_hs = [torch.randn(d) for _ in range(L + 1)]

    easy_sims = SimilarityComputer.compute(easy_hs)
    hard_sims = SimilarityComputer.compute(hard_hs)

    print(f"Easy query  — mean r_l: {easy_sims.mean():.4f}  (expected ≈ 1.0)")
    print(f"Hard query  — mean r_l: {hard_sims.mean():.4f}  (expected ≈ 0.0)")
    assert easy_sims.mean() > hard_sims.mean(), "Sanity check failed"
    print("SimilarityComputer: unit test PASSED")

_test_similarity_computer()

---
## Cell 5 — Module: Settling Layer Detector
Finds `l*` — the earliest layer after which all remaining `r_l > τ`.

In [ ]:
class SettlingDetector:
    """
    Identifies the settling layer l* from a similarity sequence.

    l* = min { l >= l_min  |  r_l' > τ  for all l' in [l, L-1] }

    If no such l exists, l* = L (the representation never settles).
    """

    def __init__(self, tau: float, l_min: int):
        """
        Parameters
        ----------
        tau   : settling threshold — r_l must exceed this for all remaining layers
        l_min : minimum layer index considered for settling (depth floor)
        """
        assert 0.0 < tau < 1.0, "tau must be in (0, 1)"
        assert l_min >= 0
        self.tau   = tau
        self.l_min = l_min

    def detect(self, similarities: np.ndarray) -> int:
        """
        Parameters
        ----------
        similarities : np.ndarray shape (L,)  — r_l values

        Returns
        -------
        int  — the settling layer l* in [l_min, L]
        """
        L = len(similarities)

        # Precompute: for each position l, does the suffix [l, L-1] all exceed tau?
        # Scan from right to left for O(L) efficiency.
        suffix_all_above = np.zeros(L, dtype=bool)
        suffix_all_above[-1] = similarities[-1] > self.tau
        for l in range(L - 2, -1, -1):
            suffix_all_above[l] = (similarities[l] > self.tau) and suffix_all_above[l + 1]

        # Find the earliest l >= l_min where the suffix condition holds
        for l in range(self.l_min, L):
            if suffix_all_above[l]:
                return l

        # Never settled — return L
        return L

    def detect_batch(
        self, batch_similarities: List[np.ndarray]
    ) -> List[int]:
        """Apply detect() to each element of a batch."""
        return [self.detect(sims) for sims in batch_similarities]


# ── Unit test ──────────────────────────────────────────────────────────────────
def _test_settling_detector():
    detector = SettlingDetector(tau=0.97, l_min=3)

    # Case 1: settles at layer 10
    sims = np.array([0.5, 0.6, 0.7, 0.8, 0.85, 0.88, 0.90, 0.92, 0.95, 0.96,
                     0.98, 0.99, 0.99, 0.99, 0.99, 0.99])
    l_star = detector.detect(sims)
    print(f"Case 1 (settles at 10)  → l* = {l_star}")
    assert l_star == 10

    # Case 2: never settles (all sims below tau)
    sims_never = np.full(16, 0.80)
    l_star_never = detector.detect(sims_never)
    print(f"Case 2 (never settles)  → l* = {l_star_never}  (= L = {len(sims_never)})")
    assert l_star_never == len(sims_never)

    # Case 3: settles at l_min (first eligible layer)
    sims_early = np.array([0.5, 0.5, 0.5, 0.99, 0.99, 0.99, 0.99, 0.99])
    l_star_early = detector.detect(sims_early)
    print(f"Case 3 (settles at l_min=3) → l* = {l_star_early}")
    assert l_star_early == 3

    print("SettlingDetector: all tests PASSED")

_test_settling_detector()

---
## Cell 6 — Module: RSD Scorer
Computes the normalized difficulty score `d_RSD = l* / L`.

In [ ]:
class RSDScorer:
    """
    End-to-end RSD scorer: query string → difficulty scalar d ∈ [0, 1].

    Composes: RoutingModel → SimilarityComputer → SettlingDetector → normalize.
    """

    def __init__(self, routing_model: RoutingModel, cfg: RSDConfig):
        self.routing_model = routing_model
        self.cfg           = cfg
        self.sim_computer  = SimilarityComputer()
        self.detector      = SettlingDetector(
            tau=cfg.tau,
            l_min=routing_model.l_min
        )
        self.L = routing_model.num_layers

    def score(self, query: str) -> Dict:
        """
        Score a single query.

        Returns
        -------
        dict with keys:
            d_rsd        : float  normalized difficulty in [0, 1]
            l_star       : int    settling layer
            similarities : array  r_l values across all layers
        """
        hidden_states = self.routing_model.extract_hidden_states(query)
        similarities  = self.sim_computer.compute(hidden_states)
        l_star        = self.detector.detect(similarities)
        d_rsd         = l_star / self.L

        return {
            "d_rsd":        d_rsd,
            "l_star":       l_star,
            "similarities": similarities,
        }

    def score_batch(
        self, queries: List[str], batch_size: int = 8
    ) -> List[Dict]:
        """
        Score a list of queries in mini-batches.

        Parameters
        ----------
        queries    : list of query strings
        batch_size : number of queries per GPU forward pass

        Returns
        -------
        List of result dicts (same structure as score())
        """
        all_results = []
        for i in tqdm(range(0, len(queries), batch_size), desc="Scoring"):
            batch = queries[i : i + batch_size]
            batch_hs   = self.routing_model.extract_hidden_states_batch(batch)
            batch_sims = self.sim_computer.compute_batch(batch_hs)
            batch_l    = self.detector.detect_batch(batch_sims)

            for sims, l_star in zip(batch_sims, batch_l):
                all_results.append({
                    "d_rsd":        l_star / self.L,
                    "l_star":       l_star,
                    "similarities": sims,
                })
        return all_results


print("RSDScorer class defined.")
print("Usage:  scorer = RSDScorer(routing_model, CFG)")
print("        result = scorer.score('What is the capital of France?')")
print("        print(result['d_rsd'])")

---
## Cell 7 — Module: Router
Maps `d_RSD` to a routing decision using threshold δ.

In [ ]:
class RSDRouter:
    """
    Maps RSD scores to routing decisions.

    Decision rule
    -------------
        d_rsd < δ  →  route to WEAK (cheap) model
        d_rsd ≥ δ  →  route to STRONG (expensive) model
    """

    WEAK   = "weak_model"
    STRONG = "strong_model"

    def __init__(self, delta: float):
        assert 0.0 <= delta <= 1.0
        self.delta = delta

    def route(self, d_rsd: float) -> str:
        """Return WEAK or STRONG for a single score."""
        return self.WEAK if d_rsd < self.delta else self.STRONG

    def route_batch(self, scores: List[float]) -> List[str]:
        """Return routing decisions for a batch of scores."""
        return [self.route(s) for s in scores]

    def strong_model_fraction(self, scores: List[float]) -> float:
        """Fraction of queries routed to the strong model."""
        decisions = self.route_batch(scores)
        return decisions.count(self.STRONG) / len(decisions)


# ── Demo with synthetic scores ─────────────────────────────────────────────────
router = RSDRouter(delta=CFG.delta)
demo_scores = [0.1, 0.3, 0.5, 0.7, 0.9]
demo_decisions = router.route_batch(demo_scores)

print("Demo routing decisions:")
for score, decision in zip(demo_scores, demo_decisions):
    label = "→ WEAK   (cheap)" if decision == RSDRouter.WEAK else "→ STRONG (expensive)"
    print(f"  d_RSD = {score:.1f}  {label}")
print(f"\nStrong model fraction: {router.strong_model_fraction(demo_scores):.0%}")

---
## Cell 8 — Threshold Calibration: τ (Settling Sensitivity)
Grid-searches τ using Spearman's ρ between d_RSD and ground-truth difficulty labels.

In [ ]:
class TauCalibrator:
    """
    Calibrates the settling threshold τ on a labeled validation set.

    Selects the τ that maximizes Spearman rank correlation between
    d_RSD and ground-truth difficulty labels.

    Ground-truth labels:
        0  =  easy  (weak model solves it correctly)
        1  =  hard  (only strong model solves it correctly)
    """

    def __init__(self, cfg: RSDConfig, l_min: int, L: int):
        self.cfg   = cfg
        self.l_min = l_min
        self.L     = L

    def calibrate(
        self,
        similarity_sequences: List[np.ndarray],  # one per validation query
        ground_truth_labels: List[int],          # 0=easy, 1=hard
    ) -> Tuple[float, Dict]:
        """
        Parameters
        ----------
        similarity_sequences : list of r_l arrays (already computed, no re-inference)
        ground_truth_labels  : binary difficulty labels

        Returns
        -------
        best_tau  : float    τ with highest Spearman ρ
        results   : dict     full grid results for plotting
        """
        results = {"tau": [], "spearman_rho": [], "roc_auc": []}

        for tau in self.cfg.tau_grid:
            detector = SettlingDetector(tau=tau, l_min=self.l_min)
            d_rsd_scores = [
                detector.detect(sims) / self.L
                for sims in similarity_sequences
            ]

            rho, _ = spearmanr(d_rsd_scores, ground_truth_labels)

            try:
                auc = roc_auc_score(ground_truth_labels, d_rsd_scores)
            except ValueError:
                auc = float("nan")

            results["tau"].append(tau)
            results["spearman_rho"].append(rho)
            results["roc_auc"].append(auc)

        best_idx = np.argmax(results["spearman_rho"])
        best_tau = results["tau"][best_idx]

        print(f"\nτ calibration results:")
        print(f"{'τ':>6}  {'Spearman ρ':>12}  {'ROC-AUC':>9}")
        for t, rho, auc in zip(results["tau"], results["spearman_rho"], results["roc_auc"]):
            marker = " ← best" if t == best_tau else ""
            print(f"{t:6.2f}  {rho:12.4f}  {auc:9.4f}{marker}")

        return best_tau, results


# ── Demo with synthetic data ───────────────────────────────────────────────────
def _demo_tau_calibration():
    rng = np.random.default_rng(42)
    L, N = 32, 300
    l_min = int(0.10 * L)

    # Simulate easy queries: high similarity throughout
    easy_sims = [np.clip(rng.normal(0.99, 0.01, L), 0, 1) for _ in range(N // 2)]
    # Simulate hard queries: low similarity until late layers
    hard_sims = [
        np.concatenate([
            np.clip(rng.normal(0.80, 0.05, L // 2), 0, 1),
            np.clip(rng.normal(0.98, 0.005, L // 2), 0, 1)
        ])
        for _ in range(N // 2)
    ]

    all_sims   = easy_sims + hard_sims
    all_labels = [0] * (N // 2) + [1] * (N // 2)  # 0=easy, 1=hard

    calibrator = TauCalibrator(CFG, l_min=l_min, L=L)
    best_tau, results = calibrator.calibrate(all_sims, all_labels)
    print(f"\nBest τ selected: {best_tau}")
    return best_tau, results

best_tau, tau_results = _demo_tau_calibration()

---
## Cell 9 — Threshold Calibration: δ (Quality-Cost Trade-off)
Sweeps δ to trace the quality-cost frontier and picks the operating point.

In [ ]:
class DeltaCalibrator:
    """
    Calibrates the routing threshold δ on the validation set.

    Traces the quality-cost trade-off curve across the δ grid
    and identifies the operating point meeting a cost constraint.

    Quality model (simplified)
    --------------------------
        Easy query  routed to weak model   → quality = weak_model_quality
        Easy query  routed to strong model → quality = strong_model_quality
        Hard query  routed to weak model   → quality = weak_model_quality  (low)
        Hard query  routed to strong model → quality = strong_model_quality
    """

    def __init__(self, cfg: RSDConfig):
        self.cfg = cfg

    def calibrate(
        self,
        d_rsd_scores: List[float],
        ground_truth_labels: List[int],
        max_strong_fraction: float = 0.40,
    ) -> Tuple[float, Dict]:
        """
        Parameters
        ----------
        d_rsd_scores          : RSD scores for all validation queries
        ground_truth_labels   : 0=easy, 1=hard
        max_strong_fraction   : maximum allowed fraction sent to strong model

        Returns
        -------
        best_delta  : float   δ that maximizes quality within cost constraint
        results     : dict    full grid results for plotting
        """
        results = {"delta": [], "strong_fraction": [], "mean_quality": []}

        for delta in self.cfg.delta_grid:
            router    = RSDRouter(delta=delta)
            decisions = router.route_batch(d_rsd_scores)

            # Compute mean quality across queries
            qualities = []
            for label, decision in zip(ground_truth_labels, decisions):
                if decision == RSDRouter.STRONG:
                    qualities.append(self.cfg.strong_model_quality)
                else:
                    # Weak model quality depends on whether query is actually hard
                    if label == 0:  # easy
                        qualities.append(self.cfg.weak_model_quality + 0.30)
                    else:           # hard — weak model struggles
                        qualities.append(self.cfg.weak_model_quality)

            strong_frac  = decisions.count(RSDRouter.STRONG) / len(decisions)
            mean_quality = np.mean(qualities)

            results["delta"].append(delta)
            results["strong_fraction"].append(strong_frac)
            results["mean_quality"].append(mean_quality)

        # Select δ that maximizes quality while respecting cost constraint
        constrained = [
            (q, d)
            for q, d, sf in zip(
                results["mean_quality"],
                results["delta"],
                results["strong_fraction"]
            )
            if sf <= max_strong_fraction
        ]
        if not constrained:
            print("Warning: no δ satisfies the cost constraint. Returning δ=0.9.")
            return 0.9, results

        best_quality, best_delta = max(constrained, key=lambda x: x[0])
        print(f"Best δ = {best_delta:.2f}  "
              f"(mean quality = {best_quality:.3f}, "
              f"constraint: strong fraction ≤ {max_strong_fraction:.0%})")
        return best_delta, results


# ── Demo ───────────────────────────────────────────────────────────────────────
def _demo_delta_calibration():
    rng = np.random.default_rng(0)
    N = 300

    # Simulate RSD scores: easy queries cluster low, hard queries cluster high
    easy_scores = np.clip(rng.normal(0.20, 0.08, N // 2), 0, 1).tolist()
    hard_scores = np.clip(rng.normal(0.75, 0.10, N // 2), 0, 1).tolist()
    all_scores  = easy_scores + hard_scores
    all_labels  = [0] * (N // 2) + [1] * (N // 2)

    calibrator = DeltaCalibrator(CFG)
    best_delta, delta_results = calibrator.calibrate(
        all_scores, all_labels, max_strong_fraction=0.40
    )
    return best_delta, delta_results, all_scores, all_labels

best_delta, delta_results, _demo_scores, _demo_labels = _demo_delta_calibration()

---
## Cell 10 — Evaluation Module
Computes all standard routing evaluation metrics.

In [ ]:
class RSDEvaluator:
    """
    Computes standard routing evaluation metrics.

    Metrics
    -------
    spearman_rho        : rank correlation between d_RSD and ground-truth labels
    roc_auc             : area under the ROC curve (d_RSD as binary classifier)
    strong_fraction     : fraction of queries routed to the strong model
    mean_quality        : average quality across all routed queries
    mean_settling_layer : average normalized settling depth (proxy for compute saved)
    """

    def evaluate(
        self,
        d_rsd_scores:       List[float],
        ground_truth_labels: List[int],
        decisions:           List[str],
        cfg:                 RSDConfig,
    ) -> Dict:
        scores_arr = np.array(d_rsd_scores)
        labels_arr = np.array(ground_truth_labels)

        rho, p_val = spearmanr(scores_arr, labels_arr)

        try:
            auc = roc_auc_score(labels_arr, scores_arr)
        except ValueError:
            auc = float("nan")

        strong_fraction = decisions.count(RSDRouter.STRONG) / len(decisions)

        # Quality computation (same simplified model as DeltaCalibrator)
        qualities = []
        for label, decision in zip(ground_truth_labels, decisions):
            if decision == RSDRouter.STRONG:
                qualities.append(cfg.strong_model_quality)
            elif label == 0:
                qualities.append(cfg.weak_model_quality + 0.30)
            else:
                qualities.append(cfg.weak_model_quality)

        metrics = {
            "spearman_rho":        rho,
            "p_value":             p_val,
            "roc_auc":             auc,
            "strong_fraction":     strong_fraction,
            "mean_quality":        float(np.mean(qualities)),
            "mean_settling_depth": float(scores_arr.mean()),
        }

        print("\nEvaluation Results")
        print("=" * 40)
        for k, v in metrics.items():
            print(f"  {k:<25} {v:.4f}")
        return metrics


# ── Run evaluation on demo data ────────────────────────────────────────────────
evaluator = RSDEvaluator()
router_eval = RSDRouter(delta=best_delta)
decisions_eval = router_eval.route_batch(_demo_scores)

metrics = evaluator.evaluate(
    d_rsd_scores=_demo_scores,
    ground_truth_labels=_demo_labels,
    decisions=decisions_eval,
    cfg=CFG
)

---
## Cell 11 — Ablation Studies
Systematically tests the three key ablations from the paper.

In [ ]:
class AblationRunner:
    """
    Runs the three ablations described in the RSD paper.

    Ablation 1 — Minimum depth floor (l_min)
    Ablation 2 — Effect of τ on Spearman ρ  (complement to TauCalibrator)
    Ablation 3 — Routing model scale proxy   (simulate different L values)
    """

    def __init__(self, cfg: RSDConfig, L: int):
        self.cfg = cfg
        self.L   = L

    # ── Ablation 1: l_min floor ────────────────────────────────────────────────
    def ablation_l_min(
        self,
        similarity_sequences: List[np.ndarray],
        labels: List[int],
        l_min_fractions: List[float] = [0.0, 0.05, 0.10, 0.15, 0.20],
    ) -> Dict:
        """
        Measures Spearman ρ for different l_min settings at fixed τ.
        """
        results = {"l_min_frac": [], "l_min_abs": [], "spearman_rho": []}

        for frac in l_min_fractions:
            l_min   = max(0, int(frac * self.L))
            detector = SettlingDetector(tau=self.cfg.tau, l_min=l_min)
            scores  = [detector.detect(s) / self.L for s in similarity_sequences]
            rho, _  = spearmanr(scores, labels)

            results["l_min_frac"].append(frac)
            results["l_min_abs"].append(l_min)
            results["spearman_rho"].append(rho)

        print("\nAblation 1 — Minimum Depth Floor (l_min)")
        print(f"{'l_min (frac)':>14}  {'l_min (abs)':>12}  {'Spearman ρ':>12}")
        for frac, l_abs, rho in zip(
            results["l_min_frac"],
            results["l_min_abs"],
            results["spearman_rho"]
        ):
            print(f"{frac:14.2f}  {l_abs:12d}  {rho:12.4f}")

        return results

    # ── Ablation 2: τ sensitivity ──────────────────────────────────────────────
    def ablation_tau_sensitivity(
        self,
        similarity_sequences: List[np.ndarray],
        labels: List[int],
    ) -> Dict:
        """
        Measures mean d_RSD variance across the τ grid.
        High variance at a τ → τ is discriminative.
        Low variance        → τ makes little difference.
        """
        results = {"tau": [], "mean_d_rsd": [], "std_d_rsd": [], "spearman_rho": []}
        l_min = int(self.cfg.l_min_frac * self.L)

        for tau in self.cfg.tau_grid:
            detector = SettlingDetector(tau=tau, l_min=l_min)
            scores   = [detector.detect(s) / self.L for s in similarity_sequences]
            rho, _   = spearmanr(scores, labels)

            results["tau"].append(tau)
            results["mean_d_rsd"].append(np.mean(scores))
            results["std_d_rsd"].append(np.std(scores))
            results["spearman_rho"].append(rho)

        print("\nAblation 2 — τ Sensitivity")
        print(f"{'τ':>6}  {'mean d_RSD':>12}  {'std d_RSD':>10}  {'Spearman ρ':>12}")
        for t, m, s, rho in zip(
            results["tau"],
            results["mean_d_rsd"],
            results["std_d_rsd"],
            results["spearman_rho"]
        ):
            print(f"{t:6.2f}  {m:12.4f}  {s:10.4f}  {rho:12.4f}")

        return results


# ── Run both ablations on synthetic data ───────────────────────────────────────
rng = np.random.default_rng(7)
L_abl, N_abl = 32, 300

easy_sims_abl = [np.clip(rng.normal(0.99, 0.01, L_abl), 0, 1) for _ in range(N_abl // 2)]
hard_sims_abl = [
    np.concatenate([
        np.clip(rng.normal(0.78, 0.05, L_abl // 2), 0, 1),
        np.clip(rng.normal(0.98, 0.005, L_abl // 2), 0, 1)
    ])
    for _ in range(N_abl // 2)
]
all_sims_abl   = easy_sims_abl + hard_sims_abl
all_labels_abl = [0] * (N_abl // 2) + [1] * (N_abl // 2)

ablation_runner = AblationRunner(CFG, L=L_abl)
abl1_results = ablation_runner.ablation_l_min(all_sims_abl, all_labels_abl)
abl2_results = ablation_runner.ablation_tau_sensitivity(all_sims_abl, all_labels_abl)

---
## Cell 12 — Visualization
All plots needed for a research paper: similarity profiles, calibration curves, ablations.

In [ ]:
class RSDVisualizer:
    """
    Produces all figures required for the RSD paper.
    """

    STYLE = dict(figsize=(7, 4), dpi=140)

    @staticmethod
    def _base_style():
        plt.rcParams.update({
            "axes.spines.top":    False,
            "axes.spines.right":  False,
            "font.family":        "sans-serif",
            "axes.grid":          True,
            "grid.alpha":         0.3,
        })

    # ── Figure 1: Similarity profile of easy vs hard query ────────────────────
    @classmethod
    def plot_similarity_profiles(
        cls,
        easy_sims: np.ndarray,
        hard_sims: np.ndarray,
        tau: float,
        l_star_easy: int,
        l_star_hard: int,
    ):
        cls._base_style()
        fig, ax = plt.subplots(**cls.STYLE)

        layers = np.arange(len(easy_sims))
        ax.plot(layers, easy_sims, color="#2196F3", lw=2, label="Easy query")
        ax.plot(layers, hard_sims, color="#E53935", lw=2, label="Hard query")
        ax.axhline(tau, color="gray", ls="--", lw=1.2, label=f"τ = {tau}")

        ax.axvline(l_star_easy, color="#2196F3", ls=":", lw=1.5,
                   label=f"l* (easy) = {l_star_easy}")
        ax.axvline(l_star_hard, color="#E53935", ls=":", lw=1.5,
                   label=f"l* (hard) = {l_star_hard}")

        ax.set_xlabel("Layer index l")
        ax.set_ylabel("Cosine similarity r_l")
        ax.set_title("Inter-layer cosine similarity profiles")
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=9)
        plt.tight_layout()
        plt.show()

    # ── Figure 2: τ calibration curve ─────────────────────────────────────────
    @classmethod
    def plot_tau_calibration(cls, tau_results: Dict, best_tau: float):
        cls._base_style()
        fig, ax = plt.subplots(**cls.STYLE)

        ax.plot(tau_results["tau"], tau_results["spearman_rho"],
                marker="o", color="#7B1FA2", lw=2, label="Spearman ρ")
        ax.plot(tau_results["tau"], tau_results["roc_auc"],
                marker="s", color="#00897B", lw=2, ls="--", label="ROC-AUC")
        ax.axvline(best_tau, color="gray", ls=":", lw=1.5,
                   label=f"Best τ = {best_tau}")

        ax.set_xlabel("Settling threshold τ")
        ax.set_ylabel("Score")
        ax.set_title("τ calibration: Spearman ρ and ROC-AUC vs τ")
        ax.legend(fontsize=9)
        plt.tight_layout()
        plt.show()

    # ── Figure 3: Quality-cost trade-off curve ─────────────────────────────────
    @classmethod
    def plot_quality_cost_tradeoff(
        cls, delta_results: Dict, best_delta: float
    ):
        cls._base_style()
        fig, ax = plt.subplots(**cls.STYLE)

        ax.plot(
            delta_results["strong_fraction"],
            delta_results["mean_quality"],
            marker="o", color="#1565C0", lw=2
        )

        # Annotate best_delta
        for d, sf, q in zip(
            delta_results["delta"],
            delta_results["strong_fraction"],
            delta_results["mean_quality"]
        ):
            if abs(d - best_delta) < 1e-6:
                ax.scatter([sf], [q], s=120, color="#E53935", zorder=5,
                           label=f"δ = {best_delta:.2f}")

        ax.set_xlabel("Strong model call fraction (cost →)")
        ax.set_ylabel("Mean quality")
        ax.set_title("Quality-cost trade-off curve (δ sweep)")
        ax.legend(fontsize=9)
        plt.tight_layout()
        plt.show()

    # ── Figure 4: Ablation — l_min effect ─────────────────────────────────────
    @classmethod
    def plot_ablation_l_min(cls, abl_results: Dict):
        cls._base_style()
        fig, ax = plt.subplots(**cls.STYLE)

        ax.plot(abl_results["l_min_frac"], abl_results["spearman_rho"],
                marker="o", color="#FF6F00", lw=2)
        ax.set_xlabel("Minimum depth floor (l_min as fraction of L)")
        ax.set_ylabel("Spearman ρ")
        ax.set_title("Ablation 1: Effect of minimum depth floor (l_min) on signal quality")
        plt.tight_layout()
        plt.show()

    # ── Figure 5: RSD score distribution by difficulty ─────────────────────────
    @classmethod
    def plot_score_distribution(
        cls, scores: List[float], labels: List[int], delta: float
    ):
        cls._base_style()
        fig, ax = plt.subplots(**cls.STYLE)

        easy = [s for s, l in zip(scores, labels) if l == 0]
        hard = [s for s, l in zip(scores, labels) if l == 1]

        ax.hist(easy, bins=30, alpha=0.6, color="#2196F3", label="Easy queries")
        ax.hist(hard, bins=30, alpha=0.6, color="#E53935", label="Hard queries")
        ax.axvline(delta, color="black", ls="--", lw=1.5,
                   label=f"δ = {delta:.2f}")

        ax.set_xlabel("d_RSD score")
        ax.set_ylabel("Count")
        ax.set_title("Distribution of RSD scores by query difficulty")
        ax.legend(fontsize=9)
        plt.tight_layout()
        plt.show()


# ── Run all plots ──────────────────────────────────────────────────────────────
viz = RSDVisualizer()

# Figure 1: Similarity profiles
detector_viz = SettlingDetector(tau=best_tau, l_min=int(0.1 * L_abl))
sample_easy  = all_sims_abl[0]
sample_hard  = all_sims_abl[N_abl // 2]
viz.plot_similarity_profiles(
    easy_sims=sample_easy,
    hard_sims=sample_hard,
    tau=best_tau,
    l_star_easy=detector_viz.detect(sample_easy),
    l_star_hard=detector_viz.detect(sample_hard),
)

# Figure 2: τ calibration
viz.plot_tau_calibration(tau_results, best_tau)

# Figure 3: Quality-cost trade-off
viz.plot_quality_cost_tradeoff(delta_results, best_delta)

# Figure 4: l_min ablation
viz.plot_ablation_l_min(abl1_results)

# Figure 5: Score distribution
viz.plot_score_distribution(_demo_scores, _demo_labels, best_delta)

---
## Cell 13 — Full End-to-End Demo
Demonstrates the complete pipeline on example queries (no GPU required — uses synthetic hidden states if model not loaded).

In [ ]:
def run_demo_queries(
    scorer_or_none,
    router: RSDRouter,
    queries: List[str],
    L: int,
):
    """
    Runs the full pipeline on a list of demo queries.
    If scorer_or_none is None, uses synthetic similarity sequences instead.
    """
    rng = np.random.default_rng(99)

    print(f"{'Query':<55} {'d_RSD':>6}  {'l*':>4}  {'Decision'}")
    print("-" * 90)

    for i, query in enumerate(queries):
        if scorer_or_none is not None:
            result   = scorer_or_none.score(query)
            d_rsd    = result["d_rsd"]
            l_star   = result["l_star"]
        else:
            # Synthetic mode: short queries settle early, long complex ones settle late
            word_count  = len(query.split())
            complexity  = min(word_count / 20, 1.0)
            # Simulate similarity sequence
            settle_frac = 0.15 + 0.70 * complexity + rng.normal(0, 0.05)
            l_star      = int(np.clip(settle_frac, 0, 1) * L)
            d_rsd       = l_star / L

        decision = router.route(d_rsd)
        tag      = "WEAK   " if decision == RSDRouter.WEAK else "STRONG "
        print(f"{query[:54]:<55} {d_rsd:>6.3f}  {l_star:>4}  {tag}")


DEMO_QUERIES = [
    "What is the capital of France?",
    "Who wrote Hamlet?",
    "What is 2 + 2?",
    "Explain the proof of Fermat's Last Theorem step by step.",
    "Compare the epistemological assumptions of Kant and Hume with respect to synthetic a priori knowledge.",
    "Write a Python implementation of a Red-Black tree with deletion.",
    "What causes rain?",
    "Solve: integrate x^2 * sin(x) dx using integration by parts twice.",
]

demo_router = RSDRouter(delta=best_delta)

# Change to: run_demo_queries(scorer, demo_router, DEMO_QUERIES, L=routing_model.num_layers)
# if routing_model is instantiated.
run_demo_queries(
    scorer_or_none=None,   # set to scorer if model is loaded
    router=demo_router,
    queries=DEMO_QUERIES,
    L=32,
)

---
## Cell 14 — Module Summary
Quick reference for all modules and their relationships.

In [ ]:
summary = """
RSD Module Architecture
=======================

RoutingModel
  .extract_hidden_states(query)          → List[Tensor]  shape (d,) × (L+1)
  .extract_hidden_states_batch(queries)  → List[List[Tensor]]

SimilarityComputer
  .compute(hidden_states)                → np.ndarray  shape (L,)
  .compute_batch(batch_hs)               → List[np.ndarray]

SettlingDetector(tau, l_min)
  .detect(similarities)                  → int  l* in [l_min, L]
  .detect_batch(batch_sims)              → List[int]

RSDScorer(routing_model, cfg)
  .score(query)                          → dict {d_rsd, l_star, similarities}
  .score_batch(queries, batch_size)      → List[dict]

RSDRouter(delta)
  .route(d_rsd)                          → str  "weak_model" | "strong_model"
  .route_batch(scores)                   → List[str]
  .strong_model_fraction(scores)         → float

TauCalibrator(cfg, l_min, L)
  .calibrate(sims, labels)               → (best_tau, results_dict)

DeltaCalibrator(cfg)
  .calibrate(scores, labels, max_sf)     → (best_delta, results_dict)

RSDEvaluator
  .evaluate(scores, labels, decisions, cfg) → metrics_dict

AblationRunner(cfg, L)
  .ablation_l_min(sims, labels)          → results_dict
  .ablation_tau_sensitivity(sims, labels)→ results_dict

RSDVisualizer
  .plot_similarity_profiles(...)         → Figure 1
  .plot_tau_calibration(...)             → Figure 2
  .plot_quality_cost_tradeoff(...)       → Figure 3
  .plot_ablation_l_min(...)              → Figure 4
  .plot_score_distribution(...)          → Figure 5

Config: RSDConfig (all parameters in one dataclass)
"""
print(summary)